# 06 - Validar candidato LakeSP

Este notebook testa um produto `LAKESP` leve antes de qualquer download PIXC. A validação usa a geometria interna do ZIP para medir a distância real entre os 13 exutórios e as feições de água superficial/lago/reservatório disponíveis.

## Por que RiverSP foi insuficiente

Dois candidatos RiverSP foram validados por geometria interna. O primeiro, `cycle 055 pass 255 tile SA`, ficou a aproximadamente 161,7 km a 163,7 km dos exutórios. O segundo, `cycle 055 pass 227 tile SA`, melhorou bastante, mas ainda ficou a aproximadamente 24,7 km a 25,9 km. Assim, RiverSP não confirmou suporte observacional real para estes pontos nesta etapa.

## Por que testar LakeSP antes de PIXC

`LAKESP` é um produto mais leve e espacialmente estruturado, adequado para verificar se há feições de água superficial próximas aos exutórios sem baixar arquivos PIXC grandes. Se LakeSP também não indicar proximidade real, o próximo passo deve ser escolhido com cautela, usando o menor recorte ou o menor conjunto PIXC possível.

In [ ]:
from __future__ import annotations

import json
import logging
import zipfile
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString
from shapely.ops import nearest_points


In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'src' / 'check_environment.py').exists():
            return candidate
    fallback = Path.home() / 'mystorage' / 'PPGGAG1889' / 'atividade3_swot'
    if fallback.exists():
        return fallback
    raise RuntimeError('FALHA: rode este notebook dentro do repositorio atividade3_swot.')

PROJECT_ROOT = find_project_root(Path.cwd())
EXUTORIOS_CSV = PROJECT_ROOT / 'dados' / 'exutorios.csv'
RANKING_CSV = PROJECT_ROOT / 'outputs' / 'tabelas' / 'ranking_candidatos_swot_proximidade.csv'
CANDIDATOS_CSV = PROJECT_ROOT / 'outputs' / 'tabelas' / 'candidatos_passagem_teste_swot.csv'
VALIDACAO_RIVERSP_CSV = PROJECT_ROOT / 'outputs' / 'tabelas' / 'validacao_riversp_top1_ranking_exutorios.csv'
RAW_DIR = PROJECT_ROOT / 'dados' / 'raw' / 'swot' / 'lakesp'
INTERMEDIATE_DIR = PROJECT_ROOT / 'dados' / 'intermediarios' / 'swot' / 'lakesp'
OUTPUT_TABLE = PROJECT_ROOT / 'outputs' / 'tabelas' / 'validacao_lakesp_exutorios.csv'
OUTPUT_FIGURE = PROJECT_ROOT / 'outputs' / 'figuras' / '06_validar_lakesp_candidato.png'
LOG_FILE = PROJECT_ROOT / 'outputs' / 'logs' / '06_validar_lakesp_candidato.log'

for path in [RAW_DIR, INTERMEDIATE_DIR, OUTPUT_TABLE.parent, OUTPUT_FIGURE.parent, LOG_FILE.parent]:
    path.mkdir(parents=True, exist_ok=True)

logging.basicConfig(filename=LOG_FILE, filemode='w', level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
print('OK raiz do projeto:', PROJECT_ROOT)
print('OK raw:', RAW_DIR)
print('OK log:', LOG_FILE)


## Entradas

A célula abaixo lê os 13 exutórios, o ranking do notebook 04, a tabela de candidatos do notebook 02 quando disponível e a validação RiverSP top 1 quando disponível.

In [ ]:
expected_columns = ['id', 'latitude', 'longitude']
if not EXUTORIOS_CSV.exists():
    raise FileNotFoundError(f'FALHA: arquivo de exutorios nao encontrado: {EXUTORIOS_CSV}')
if not RANKING_CSV.exists():
    raise FileNotFoundError(f'FALHA: ranking do notebook 04 nao encontrado: {RANKING_CSV}')

exutorios = pd.read_csv(EXUTORIOS_CSV)
if list(exutorios.columns) != expected_columns:
    raise ValueError(f'FALHA: colunas esperadas {expected_columns}, colunas encontradas {list(exutorios.columns)}')
if len(exutorios) != 13:
    raise ValueError(f'FALHA: esperados 13 exutorios, encontrados {len(exutorios)}')
exutorios['latitude'] = pd.to_numeric(exutorios['latitude'], errors='raise')
exutorios['longitude'] = pd.to_numeric(exutorios['longitude'], errors='raise')

ranking = pd.read_csv(RANKING_CSV, dtype={'cycle': str, 'pass': str, 'tile': str})
candidatos_02 = pd.read_csv(CANDIDATOS_CSV, dtype={'cycle': str, 'pass': str, 'tile': str}) if CANDIDATOS_CSV.exists() else pd.DataFrame()
validacao_riversp = pd.read_csv(VALIDACAO_RIVERSP_CSV, dtype={'cycle': str, 'pass': str, 'tile': str}) if VALIDACAO_RIVERSP_CSV.exists() else pd.DataFrame()

print('OK exutorios:', len(exutorios))
print('OK ranking:', len(ranking))
print('OK candidatos notebook 02:', 'sim' if not candidatos_02.empty else 'nao encontrado')
print('OK validacao RiverSP top 1:', 'sim' if not validacao_riversp.empty else 'nao encontrada')
display(exutorios)


## Seleção do candidato LakeSP

A seleção parte dos candidatos `LAKESP` do ranking. Quando existir um LakeSP com o mesmo `cycle/pass` do RiverSP top 1 já testado, ele é priorizado para comparar produtos na mesma passagem. Em seguida entram menor tamanho, data mais recente e URL válida.

In [ ]:
lakesp = ranking[ranking['produto'].str.upper().eq('LAKESP')].copy()
if lakesp.empty:
    raise ValueError('FALHA: nenhum candidato LAKESP encontrado no ranking do notebook 04.')

for col in ['cycle', 'pass']:
    lakesp[col] = lakesp[col].astype(str).str.zfill(3)
lakesp['tile'] = lakesp['tile'].astype(str)
lakesp['has_url'] = lakesp['download_url'].astype(str).str.startswith('http')
lakesp['data_inicio_dt'] = pd.to_datetime(lakesp['data_inicio'], errors='coerce', utc=True)
lakesp['tamanho_mb_num'] = pd.to_numeric(lakesp['tamanho_mb'], errors='coerce')

preferred_cycle = '055'
preferred_pass = '227'
if not validacao_riversp.empty:
    preferred_cycle = str(validacao_riversp['cycle'].dropna().iloc[0]).zfill(3)
    preferred_pass = str(validacao_riversp['pass'].dropna().iloc[0]).zfill(3)

lakesp['mesmo_cycle_pass_riversp_top1'] = lakesp['cycle'].eq(preferred_cycle) & lakesp['pass'].eq(preferred_pass)
lakesp_sorted = lakesp.sort_values(
    by=['mesmo_cycle_pass_riversp_top1', 'tamanho_mb_num', 'data_inicio_dt', 'has_url'],
    ascending=[False, True, False, False],
    na_position='last',
).reset_index(drop=True)

candidate = lakesp_sorted.iloc[0].to_dict()
url = str(candidate['download_url'])
granule_id = str(candidate['granule_id'])
expected_mb = float(candidate['tamanho_mb_num']) if pd.notna(candidate.get('tamanho_mb_num')) else None
filename = Path(urlparse(url).path).name or f'{granule_id}.zip'

if not url.startswith('http'):
    raise ValueError(f'FALHA: candidato LAKESP selecionado sem URL valida: {url}')
if not filename.lower().endswith('.zip'):
    raise ValueError(f'FALHA: URL do candidato LAKESP nao aponta para .zip: {url}')

zip_path = RAW_DIR / filename
logging.info('Candidato LAKESP selecionado: %s', json.dumps(candidate, ensure_ascii=False, default=str))
print('OK candidato LAKESP selecionado')
print('rank original:', candidate.get('rank'))
print('cycle/pass/tile:', candidate.get('cycle'), candidate.get('pass'), candidate.get('tile'))
print('mesmo cycle/pass do RiverSP top1:', candidate.get('mesmo_cycle_pass_riversp_top1'))
print('granule_id:', granule_id)
print('url:', url)
print('tamanho esperado MB:', expected_mb)
print('destino:', zip_path)
display(lakesp_sorted[['rank','produto','cycle','pass','tile','data_inicio','tamanho_mb','mesmo_cycle_pass_riversp_top1','granule_id']].head(10))


## Download controlado

A próxima célula baixa somente o ZIP LakeSP selecionado, se ele ainda não existir em `dados/raw/swot/lakesp/`. Nenhum PIXC é baixado.

In [ ]:
try:
    import earthaccess
except Exception as exc:
    logging.exception('Falha ao importar earthaccess')
    raise RuntimeError('FALHA: earthaccess nao esta instalado. Rode pip install -r requirements.txt.') from exc

already_exists = zip_path.exists() and zip_path.stat().st_size > 0
if already_exists:
    downloaded_paths = [zip_path]
    status_download = 'arquivo_ja_existia'
else:
    try:
        earthaccess.login(strategy='interactive', persist=True)
        downloaded_paths = earthaccess.download(url, local_path=RAW_DIR, threads=1, show_progress=True)
        status_download = 'baixado'
    except Exception as exc:
        logging.exception('Falha no download LAKESP: %s', url)
        raise RuntimeError('FALHA: download do LAKESP nao concluido. Verifique login Earthdata, URL e conectividade.') from exc

if not downloaded_paths:
    raise RuntimeError('FALHA: earthaccess.download nao retornou caminho baixado.')
zip_path = Path(downloaded_paths[0])
if not zip_path.exists():
    candidate_path = RAW_DIR / filename
    if candidate_path.exists():
        zip_path = candidate_path
    else:
        raise FileNotFoundError(f'FALHA: arquivo baixado nao encontrado: {downloaded_paths[0]}')

downloaded_mb = zip_path.stat().st_size / (1024 * 1024)
logging.info('granule_id: %s', granule_id)
logging.info('URL usada: %s', url)
logging.info('Arquivo: %s', filename)
logging.info('Tamanho esperado MB: %s', expected_mb)
logging.info('Tamanho local MB: %.3f', downloaded_mb)
logging.info('Caminho local: %s', zip_path)
logging.info('Status download: %s', status_download)

print('OK download LAKESP:', status_download)
print('arquivo:', zip_path)
print('tamanho local MB:', round(downloaded_mb, 3))


## Inspeção do ZIP e leitura espacial

O ZIP é validado e seus arquivos internos são listados antes da leitura. O notebook aceita shapefile, GeoPackage, GeoJSON/JSON e tenta abrir NetCDF apenas como diagnóstico de formato, sem assumir esquema previamente.

In [ ]:
if not zipfile.is_zipfile(zip_path):
    raise zipfile.BadZipFile(f'FALHA: arquivo nao e um ZIP valido: {zip_path}')

with zipfile.ZipFile(zip_path) as zf:
    members = zf.infolist()
    zip_inventory = pd.DataFrame([
        {'nome': m.filename, 'tamanho_bytes': m.file_size, 'compactado_bytes': m.compress_size}
        for m in members
    ]).sort_values('nome').reset_index(drop=True)

logging.info('Arquivos internos do ZIP: %s', len(zip_inventory))
for name in zip_inventory['nome'].tolist():
    logging.info('ZIP member: %s', name)

vector_ext = ('.shp', '.gpkg', '.geojson', '.json')
netcdf_ext = ('.nc', '.nc4')
table_ext = ('.csv', '.parquet')
vector_members = [n for n in zip_inventory['nome'] if n.lower().endswith(vector_ext)]
netcdf_members = [n for n in zip_inventory['nome'] if n.lower().endswith(netcdf_ext)]
table_members = [n for n in zip_inventory['nome'] if n.lower().endswith(table_ext)]

print('OK arquivos no ZIP:', len(zip_inventory))
print('vetoriais candidatos:', vector_members)
print('netCDF candidatos:', netcdf_members)
print('tabelares candidatos:', table_members)
display(zip_inventory)


In [ ]:
def member_priority(name: str) -> tuple[int, str]:
    lower = name.lower()
    if lower.endswith('.shp') and any(token in lower for token in ['obs', 'prior', 'lake', 'water']):
        return (0, lower)
    if lower.endswith('.shp'):
        return (1, lower)
    if lower.endswith(('.gpkg', '.geojson')):
        return (2, lower)
    if lower.endswith('.json'):
        return (3, lower)
    return (9, lower)

if not vector_members:
    raise RuntimeError(f'FALHA: nenhum arquivo vetorial reconhecido dentro do ZIP LAKESP. NetCDF encontrados: {netcdf_members}')

selected_member = sorted(vector_members, key=member_priority)[0]
logging.info('Arquivo vetorial selecionado para leitura: %s', selected_member)
print('OK camada/arquivo selecionado:', selected_member)

read_errors = []
lakesp_gdf = None
try:
    lakesp_gdf = gpd.read_file(f'zip://{zip_path}!{selected_member}')
except Exception as exc:
    read_errors.append(f'zip:// falhou: {exc}')

if lakesp_gdf is None:
    extract_dir = INTERMEDIATE_DIR / zip_path.stem
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        related_prefix = Path(selected_member).with_suffix('').name
        for member in zf.namelist():
            base = Path(member).with_suffix('').name
            if base == related_prefix:
                zf.extract(member, extract_dir)
    extracted_candidates = list(extract_dir.rglob(Path(selected_member).name))
    if not extracted_candidates:
        raise RuntimeError('FALHA: nao foi possivel extrair o arquivo vetorial selecionado.')
    try:
        lakesp_gdf = gpd.read_file(extracted_candidates[0])
    except Exception as exc:
        read_errors.append(f'extracao falhou: {exc}')
        logging.error('Erros de leitura: %s', read_errors)
        raise RuntimeError('FALHA: nao foi possivel ler a camada LAKESP com geopandas.') from exc

if lakesp_gdf.empty:
    raise RuntimeError('FALHA: camada LAKESP lida, mas sem feicoes.')
if lakesp_gdf.crs is None:
    lakesp_gdf = lakesp_gdf.set_crs('EPSG:4326', allow_override=True)

logging.info('CRS LAKESP: %s', lakesp_gdf.crs)
logging.info('Total feicoes LAKESP: %s', len(lakesp_gdf))
logging.info('Colunas LAKESP: %s', list(lakesp_gdf.columns))
print('OK LAKESP carregado')
print('CRS:', lakesp_gdf.crs)
print('feicoes:', len(lakesp_gdf))
print('colunas:', list(lakesp_gdf.columns))
display(lakesp_gdf.head())


## Critério de suporte preliminar

O limiar inicial é `500 m`. O suporte `sim` exige proximidade real entre o exutório e uma feição LakeSP. Distâncias maiores são classificadas como `nao`; ausência de identificador espacial claro ou problema de leitura gera `indeterminado`.

In [ ]:
SUPPORT_DISTANCE_M = 500
METRIC_CRS = 'EPSG:32723'

points_gdf = gpd.GeoDataFrame(
    exutorios.copy(),
    geometry=gpd.points_from_xy(exutorios['longitude'], exutorios['latitude']),
    crs='EPSG:4326',
)
points_m = points_gdf.to_crs(METRIC_CRS)
lakesp_m = lakesp_gdf.to_crs(METRIC_CRS)

id_candidates = ['lake_id', 'obs_id', 'prior_id', 'reach_id', 'node_id', 'id', 'feature_id', 'water_id']
name_candidates = ['lake_name', 'name', 'water_body_name', 'river_name', 'p_name', 'obs_name']
id_columns = [c for c in id_candidates if c in lakesp_m.columns]
name_columns = [c for c in name_candidates if c in lakesp_m.columns]
attribute_columns = [c for c in ['lake_id', 'obs_id', 'prior_id', 'lake_name', 'water_body_name', 'wse', 'area_total', 'area_detct', 'quality_f', 'dark_frac', 'geometry'] if c in lakesp_m.columns]
if not attribute_columns:
    attribute_columns = list(lakesp_m.columns[: min(10, len(lakesp_m.columns))])

if not id_columns:
    logging.warning('Nenhuma coluna identificadora padrao encontrada no LAKESP.')

rows = []
nearest_lines = []
for _, point in points_m.iterrows():
    distances = lakesp_m.geometry.distance(point.geometry)
    if distances.empty or distances.isna().all():
        rows.append({
            'id': point['id'], 'latitude': point['latitude'], 'longitude': point['longitude'],
            'cycle': str(candidate['cycle']).zfill(3), 'pass': str(candidate['pass']).zfill(3), 'tile': str(candidate['tile']), 'granule_id': granule_id,
            'distancia_m_feicao_lakesp': pd.NA, 'feicao_mais_proxima_id': '', 'water_body_name': '',
            'suporte_lakesp': 'indeterminado', 'observacoes': 'Nao foi possivel calcular distancia ate feicoes LAKESP.',
        })
        continue

    nearest_idx = distances.idxmin()
    nearest = lakesp_m.loc[nearest_idx]
    dist_m = float(distances.loc[nearest_idx])

    feature_id = ''
    for col in id_columns:
        value = nearest.get(col)
        if pd.notna(value):
            feature_id = str(value)
            break

    water_body_name = ''
    for col in name_columns:
        value = nearest.get(col)
        if pd.notna(value):
            water_body_name = str(value)
            break

    attrs = {col: nearest.get(col) for col in attribute_columns if col != 'geometry'}
    if not id_columns:
        support = 'indeterminado'
        reason = 'sem coluna identificadora padrao; suporte mantido indeterminado'
    elif dist_m <= SUPPORT_DISTANCE_M:
        support = 'sim'
        reason = f'distancia <= {SUPPORT_DISTANCE_M} m'
    else:
        support = 'nao'
        reason = f'distancia > {SUPPORT_DISTANCE_M} m'

    geom = lakesp_m.geometry.loc[nearest_idx]
    try:
        nearest_point = nearest_points(point.geometry, geom)[1]
        nearest_lines.append({'id': point['id'], 'geometry': LineString([point.geometry, nearest_point])})
    except Exception as exc:
        logging.warning('Nao foi possivel criar linha de proximidade para %s: %s', point['id'], exc)

    rows.append({
        'id': point['id'],
        'latitude': point['latitude'],
        'longitude': point['longitude'],
        'cycle': str(candidate['cycle']).zfill(3),
        'pass': str(candidate['pass']).zfill(3),
        'tile': str(candidate['tile']),
        'granule_id': granule_id,
        'distancia_m_feicao_lakesp': round(dist_m, 2),
        'feicao_mais_proxima_id': feature_id,
        'water_body_name': water_body_name,
        'suporte_lakesp': support,
        'observacoes': f'{reason}; attrs: {attrs}',
    })

validation = pd.DataFrame(rows)
validation.to_csv(OUTPUT_TABLE, index=False, encoding='utf-8')
logging.info('Tabela de validacao LAKESP salva: %s', OUTPUT_TABLE)
logging.info('Suporte LAKESP: %s', validation['suporte_lakesp'].value_counts(dropna=False).to_dict())
print('OK tabela salva:', OUTPUT_TABLE)
print(validation['suporte_lakesp'].value_counts(dropna=False))
display(validation)


## Comparação com RiverSP

A comparação abaixo contextualiza LakeSP contra os dois testes RiverSP já realizados: `pass 255`, com cerca de 161,7 km a 163,7 km, e `pass 227`, com cerca de 24,7 km a 25,9 km.

In [ ]:
current_dist = validation['distancia_m_feicao_lakesp'].dropna()
comparison = {
    'riversp_pass_255_distancia_km': '161.7 a 163.7',
    'riversp_pass_227_distancia_km': '24.7 a 25.9',
    'lakesp_cycle': str(candidate['cycle']).zfill(3),
    'lakesp_pass': str(candidate['pass']).zfill(3),
    'lakesp_tile': str(candidate['tile']),
    'lakesp_distancia_min_km': round(current_dist.min() / 1000, 2) if not current_dist.empty else None,
    'lakesp_distancia_mediana_km': round(current_dist.median() / 1000, 2) if not current_dist.empty else None,
    'lakesp_distancia_max_km': round(current_dist.max() / 1000, 2) if not current_dist.empty else None,
    'lakesp_melhora_sobre_riversp_227': 'sim' if not current_dist.empty and (current_dist.median() / 1000) < 24.7 else 'nao',
    'suporte_lakesp': validation['suporte_lakesp'].value_counts(dropna=False).to_dict(),
}
logging.info('Comparacao LAKESP vs RiverSP: %s', json.dumps(comparison, ensure_ascii=False))
print(json.dumps(comparison, ensure_ascii=False, indent=2))


## Visualização

A figura mostra os exutórios, as feições LakeSP carregadas e linhas até a feição mais próxima quando possível.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
try:
    lakesp_plot = lakesp_gdf.to_crs('EPSG:4326')
    points_plot = points_gdf.to_crs('EPSG:4326')
    lines_plot = gpd.GeoDataFrame(nearest_lines, crs=METRIC_CRS).to_crs('EPSG:4326') if nearest_lines else gpd.GeoDataFrame(geometry=[], crs='EPSG:4326')

    minx, miny, maxx, maxy = points_plot.total_bounds
    pad_x = max((maxx - minx) * 4, 0.04)
    pad_y = max((maxy - miny) * 4, 0.04)

    lakesp_plot.plot(ax=ax, color='#41ab5d', edgecolor='#006d2c', linewidth=1.0, alpha=0.55, label='LakeSP candidato')
    if not lines_plot.empty:
        lines_plot.plot(ax=ax, color='#fb6a4a', linewidth=0.8, alpha=0.65, label='Ligacao ate feicao mais proxima')
    points_plot.plot(ax=ax, color='black', markersize=48, label='Exutorios', zorder=4)
    for _, row in points_plot.iterrows():
        ax.annotate(row['id'], (row.geometry.x, row.geometry.y), xytext=(4, 4), textcoords='offset points', fontsize=8)

    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)
    ax.set_title('Validacao LakeSP - distancias reais')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(OUTPUT_FIGURE, dpi=180)
    logging.info('Figura salva: %s', OUTPUT_FIGURE)
    print('OK figura salva:', OUTPUT_FIGURE)
    plt.show()
except Exception as exc:
    logging.exception('Falha ao gerar figura LAKESP')
    raise RuntimeError('FALHA: nao foi possivel gerar a figura de validacao LAKESP.') from exc


## Interpretação da tabela final

- `suporte_lakesp = sim`: há feição LakeSP a até 500 m do exutório.
- `suporte_lakesp = nao`: a feição LakeSP mais próxima está distante demais para suporte preliminar.
- `suporte_lakesp = indeterminado`: a leitura, a geometria ou os atributos não permitem conclusão conservadora.

Essa validação testa proximidade espacial real, não qualidade hidrológica final.

## Recomendação do próximo passo

Se LakeSP confirmar proximidade real, usar esse candidato para uma inspeção temporal/espacial leve. Se LakeSP também ficar distante, revisar os próximos candidatos leves do ranking e só considerar PIXC com um alvo espacial bem delimitado, evitando download amplo.